<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-02-embeddings-and-vectors/lesson-2.2-embeddings/practice/GCP_Capstone_2.2_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 2.2 — Embeddings — Text to Vectors

8 hands-on exercises with complete solutions. Master embedding APIs, cosine similarity, task types, cross-language search, and build a production embeddings module.

Runnable companion to the published practice lab. Each exercise below shows the objective and a complete solution. Cloud Shell / `gcloud` steps are `%%bash` cells; Python steps run in Colab after you authenticate and set your project.

---

## Exercise 1: First Embedding — 5 Sentences  
**Difficulty:** Easy

Embed 5 different sentences with gemini-embedding-001. Print dimensions, first 5 values, and token count.

1. Initialize genai.Client with Vertex AI
2. Call embed_content with 5 sentences
3. Print dimensions, first 5 values, and token count for each

**Solution:**

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(enterprise=True, project="YOUR-PROJECT", location="us-central1")

texts = [
    "Machine learning automates analytical model building.",
    "Hyderabad is famous for its biryani and pearls.",
    "Docker containers isolate application dependencies.",
    "The Ganges river flows through northern India.",
    "GPT and Gemini are large language models.",
]

# gemini-embedding-001 takes ONE text per call — loop the list
embeddings = []
for text in texts:
    resp = client.models.embed_content(
        model="gemini-embedding-001", contents=text,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_DOCUMENT", output_dimensionality=768),
    )
    embeddings.append(resp.embeddings[0])

for i, emb in enumerate(embeddings):
    v = emb.values
    print(f"  Doc {i}: {len(v)} dims | {emb.statistics.token_count} tok | [{v[0]:.4f}, {v[1]:.4f}, {v[2]:.4f}, ...]")

## Exercise 2: Cosine Similarity Matrix  
**Difficulty:** Easy

Embed 4 texts: 2 about passwords, 2 about weather. Build a similarity matrix. Verify similar pairs > 0.80.

1. Define 4 texts: 2 password-related, 2 weather-related
2. Embed all 4 with SEMANTIC_SIMILARITY
3. Compute pairwise cosine similarity
4. Verify password pair > 0.80, cross-topic pairs < 0.50

**Solution:**

In [ ]:
import numpy as np

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

texts = [
    "How do I reset my password?",
    "I forgot my login credentials",
    "What is the weather like today?",
    "Is it going to rain tomorrow?",
]

# gemini-embedding-001 takes ONE text per call — loop the list
vecs = []
for text in texts:
    resp = client.models.embed_content(
        model="gemini-embedding-001", contents=text,
        config=types.EmbedContentConfig(
            task_type="SEMANTIC_SIMILARITY", output_dimensionality=768),
    )
    vecs.append(resp.embeddings[0].values)

print("Similarity Matrix:")
for i in range(len(texts)):
    for j in range(i+1, len(texts)):
        sim = cosine_sim(vecs[i], vecs[j])
        print(f"  {sim:.4f} | {texts[i][:30]} <-> {texts[j][:30]}")

## Exercise 3: Model Comparison  
**Difficulty:** Easy

Embed the same text with text-embedding-005 and gemini-embedding-001 at 768d. Compare outputs.

1. Choose a sample text
2. Embed with both models at output_dimensionality=768
3. Compare: dimensions, token counts, first values
4. Try computing cosine similarity between the two — explain why it is meaningless

**Solution:**

In [ ]:
text = "Transformers revolutionized natural language processing."

for model in ["text-embedding-005", "gemini-embedding-001"]:
    r = client.models.embed_content(
        model=model, contents=text,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_DOCUMENT", output_dimensionality=768),
    )
    v = r.embeddings[0].values
    tok = r.embeddings[0].statistics.token_count
    print(f"  {model:<25} {len(v)} dims | {tok} tokens | [{v[0]:.4f}, {v[1]:.4f}]")

# Cross-model similarity is MEANINGLESS
sim = cosine_sim(
    client.models.embed_content(model="text-embedding-005", contents=text,
        config={"task_type":"RETRIEVAL_DOCUMENT","output_dimensionality":768}).embeddings[0].values,
    client.models.embed_content(model="gemini-embedding-001", contents=text,
        config={"task_type":"RETRIEVAL_DOCUMENT","output_dimensionality":768}).embeddings[0].values,
)
print(f"\n  Cross-model similarity: {sim:.4f} (MEANINGLESS — different spaces!)")

## Exercise 4: Task Type Impact Test  
**Difficulty:** Medium

Embed the same document+query pair with asymmetric RETRIEVAL vs symmetric SEMANTIC_SIMILARITY. Measure the difference.

1. Choose a document and a question about it
2. Embed with RETRIEVAL_DOCUMENT + RETRIEVAL_QUERY (correct)
3. Embed with SEMANTIC_SIMILARITY for both (wrong)
4. Compare cosine similarity scores

**Solution:**

In [ ]:
doc = "The aurora borealis occurs when solar wind particles collide with atmospheric gases near Earth's magnetic poles."
query = "What causes the northern lights?"

# Correct: asymmetric
d_ret = client.models.embed_content(model="gemini-embedding-001", contents=doc,
    config={"task_type":"RETRIEVAL_DOCUMENT","output_dimensionality":768}).embeddings[0].values
q_ret = client.models.embed_content(model="gemini-embedding-001", contents=query,
    config={"task_type":"RETRIEVAL_QUERY","output_dimensionality":768}).embeddings[0].values

# Wrong: symmetric
d_sem = client.models.embed_content(model="gemini-embedding-001", contents=doc,
    config={"task_type":"SEMANTIC_SIMILARITY","output_dimensionality":768}).embeddings[0].values
q_sem = client.models.embed_content(model="gemini-embedding-001", contents=query,
    config={"task_type":"SEMANTIC_SIMILARITY","output_dimensionality":768}).embeddings[0].values

print(f"  Asymmetric (correct): {cosine_sim(d_ret, q_ret):.4f}")
print(f"  Symmetric (wrong):    {cosine_sim(d_sem, q_sem):.4f}")
print(f"  Difference:           {cosine_sim(d_ret, q_ret) - cosine_sim(d_sem, q_sem):.4f}")

## Exercise 5: Mini Search Engine  
**Difficulty:** Medium

Build a 10-document semantic search engine with ranked results.

1. Define 10 diverse documents
2. Embed all with RETRIEVAL_DOCUMENT
3. Embed 3 test queries with RETRIEVAL_QUERY
4. Rank and display top-3 results for each query

**Solution:**

In [ ]:
import numpy as np

corpus = [
    "Python is a popular programming language for data science.",
    "React is a JavaScript library for building user interfaces.",
    "Gradient descent optimizes neural network weights iteratively.",
    "Hyderabad is home to major tech companies and startups.",
    "Docker containers package apps with all their dependencies.",
    "The transformer architecture uses multi-head self-attention.",
    "Kubernetes orchestrates container deployment at scale.",
    "RAG combines document retrieval with language model generation.",
    "Mumbai is India's financial capital with the BSE stock exchange.",
    "Fine-tuning adapts a pre-trained model to a specific domain.",
]

# Embed corpus — gemini-embedding-001 takes ONE text per call, so loop
doc_vecs = []
for text in corpus:
    d_r = client.models.embed_content(
        model="gemini-embedding-001", contents=text,
        config={"task_type":"RETRIEVAL_DOCUMENT","output_dimensionality":768})
    doc_vecs.append(d_r.embeddings[0].values)
doc_vecs = np.array(doc_vecs)

# Normalize
doc_vecs = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)

queries = ["How does attention work in neural networks?",
           "Which Indian city is best for tech jobs?",
           "How to deploy containers in production?"]

for query in queries:
    q_r = client.models.embed_content(
        model="gemini-embedding-001", contents=query,
        config={"task_type":"RETRIEVAL_QUERY","output_dimensionality":768})
    q_vec = np.array(q_r.embeddings[0].values)
    q_vec = q_vec / np.linalg.norm(q_vec)
    
    sims = doc_vecs @ q_vec
    top3 = np.argsort(sims)[::-1][:3]
    
    print(f"\nQuery: {query}")
    for rank, idx in enumerate(top3):
        print(f"  {rank+1}. [{sims[idx]:.4f}] {corpus[idx]}")

## Exercise 6: Cross-Language Search  
**Difficulty:** Medium

Embed equivalent sentences in English, Hindi, Telugu. Verify cross-language similarity > 0.75.

1. Write 3 parallel sentences in English + Hindi + Telugu
2. Embed all 9 with SEMANTIC_SIMILARITY
3. Compute cross-language similarity for each parallel pair
4. Compare same-topic cross-language vs different-topic same-language

**Solution:**

In [ ]:
pairs = [
    ("AI is transforming healthcare", "कृत्रिम बुद्धिमत्ता स्वास्थ्य सेवा को बदल रही है", "EN-HI: healthcare"),
    ("Python is great for data science", "पायथन डेटा साइंस के लिए बहुत अच्छा है", "EN-HI: python"),
    ("The weather is sunny today", "आज मौसम धूप वाला है", "EN-HI: weather"),
]

all_texts = []
for en, hi, label in pairs:
    all_texts.extend([en, hi])

# gemini-embedding-001 takes ONE text per call — loop the list
vecs = []
for text in all_texts:
    resp = client.models.embed_content(
        model="gemini-embedding-001", contents=text,
        config={"task_type":"SEMANTIC_SIMILARITY","output_dimensionality":768})
    vecs.append(resp.embeddings[0].values)

print("Cross-Language Similarity (English <-> Hindi):")
for i, (en, hi, label) in enumerate(pairs):
    sim = cosine_sim(vecs[i*2], vecs[i*2+1])
    print(f"  {sim:.4f} | {label}")

## Exercise 7: Matryoshka Quality Test  
**Difficulty:** Challenge

Embed a 10-doc corpus at 128, 256, 768, 3072 dims. Run same queries. Compare retrieval accuracy at each dimension.

1. Define 10 documents and 3 queries with known correct answers
2. Embed corpus at each dimension size
3. For each dimension, find top-1 result for each query
4. Check if top-1 matches the expected correct document

**Solution:**

In [ ]:
corpus = [
    "Python uses indentation for code blocks.",
    "JavaScript runs natively in web browsers.",
    "The Taj Mahal is in Agra, Uttar Pradesh.",
    "Gradient descent minimizes loss functions.",
    "Hyderabad biryani uses basmati rice and spices.",
]
queries = [("Best language for web development?", 1),
           ("Famous monuments in India?", 2),
           ("How do neural networks learn?", 3)]

for dims in [128, 256, 768, 3072]:
    # gemini-embedding-001 takes ONE text per call — loop the corpus
    d_vecs = []
    for text in corpus:
        d_r = client.models.embed_content(
            model="gemini-embedding-001", contents=text,
            config={"task_type":"RETRIEVAL_DOCUMENT","output_dimensionality":dims})
        d_vecs.append(d_r.embeddings[0].values)
    d_vecs = np.array(d_vecs)
    d_vecs = d_vecs / np.linalg.norm(d_vecs, axis=1, keepdims=True)
    
    correct = 0
    for q_text, expected_idx in queries:
        q_r = client.models.embed_content(
            model="gemini-embedding-001", contents=q_text,
            config={"task_type":"RETRIEVAL_QUERY","output_dimensionality":dims})
        q_v = np.array(q_r.embeddings[0].values)
        q_v = q_v / np.linalg.norm(q_v)
        top = np.argmax(d_vecs @ q_v)
        if top == expected_idx: correct += 1
    print(f"  {dims:>5}d: {correct}/{len(queries)} correct ({correct/len(queries)*100:.0f}%)")

## Exercise 8: Production Embeddings Module  
**Difficulty:** Challenge

Build the complete embeddings.py module with embed_documents(), embed_query(), search(), and normalize(). Test end-to-end.

1. Define embed_documents() with batching and normalization
2. Define embed_query() with normalization
3. Define search() with top-k ranking
4. Test with 20 documents and 5 queries

**Solution:**

In [ ]:
"""Embeddings module for DocuMind AI Capstone."""
from google import genai
from google.genai import types
import numpy as np

MODEL = "gemini-embedding-001"
DIMS = 768

def normalize(vec):
    v = np.array(vec)
    return (v / np.linalg.norm(v)).tolist()

def embed_documents(client, texts, batch_size=100):
    # gemini-embedding-001 accepts ONE text per call on Vertex AI, so we
    # loop; batch_size only controls how often progress is logged.
    all_vecs = []
    for i, text in enumerate(texts):
        r = client.models.embed_content(
            model=MODEL, contents=text,
            config=types.EmbedContentConfig(
                task_type="RETRIEVAL_DOCUMENT", output_dimensionality=DIMS))
        all_vecs.append(normalize(r.embeddings[0].values))
        if (i + 1) % batch_size == 0:
            print(f"  embedded {i + 1}/{len(texts)}")
    return all_vecs

def embed_query(client, query):
    r = client.models.embed_content(
        model=MODEL, contents=query,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY", output_dimensionality=DIMS))
    return normalize(r.embeddings[0].values)

def search(query_vec, doc_vecs, texts, top_k=3):
    q = np.array(query_vec)
    d = np.array(doc_vecs)
    sims = d @ q  # dot product = cosine for normalized
    top = np.argsort(sims)[::-1][:top_k]
    return [(texts[i], float(sims[i])) for i in top]

# Test
if __name__ == "__main__":
    client = genai.Client(enterprise=True, project="YOUR-PROJECT", location="us-central1")
    docs = ["Python is great for ML", "Hyderabad has amazing food",
            "Docker simplifies deployment", "Transformers use attention"]
    vecs = embed_documents(client, docs)
    q = embed_query(client, "How does deep learning work?")
    results = search(q, vecs, docs)
    for text, score in results:
        print(f"  [{score:.4f}] {text}")